In [0]:
# --- Configuration & Error Settings ---
S3_BUCKET = "s3://zemoso-s3-poc"
SOURCE_PATH = f"{S3_BUCKET}/data/source/"
SCHEMA_LOCATION = f"{S3_BUCKET}/data/schemas/bronze"
CHECKPOINT_LOCATION = f"{S3_BUCKET}/data/checkpoints/bronze_v2"
BRONZE_S3_PATH = f"{S3_BUCKET}/bronze/consumption"
BAD_RECORDS_PATH = f"{S3_BUCKET}/data/bad_records/bronze/"


In [0]:
# --- Imports ---
import traceback
from pyspark.sql.functions import col, regexp_extract, current_timestamp, upper, when, lit
from pyspark.sql.utils import AnalysisException


In [0]:
# --- Logging Setup ---
# In production, use Python's built-in logging module writing to CloudWatch or Databricks logs
import logging
logger = logging.getLogger("BronzeETL")
logger.setLevel(logging.INFO)


In [0]:
# --- Process Execution ---
try:
    logger.info(f"Starting Bronze Ingestion from {SOURCE_PATH}")

    # --- Month Mapping ---
    month_map = {
        "JANUARY": 1, "FEBRUARY": 2, "MARCH": 3, "MAR": 3, "APRIL": 4, "MAY": 5, "JUNE": 6,
        "JULY": 7, "AUGUST": 8, "SEPTEMBER": 9, "OCTOBER": 10, "NOVEMBER": 11, "DECEMBER": 12
    }

    # --- 1. Robust Ingestion Stream (AutoLoader with schema enforcement) ---
    # NOTE: Added 'badRecordsPath' to capture unparseable CSV rows without crashing the cluster
    bronze_df = (spark.readStream.format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("cloudFiles.schemaLocation", SCHEMA_LOCATION)
        .option("cloudFiles.schemaEvolutionMode", "addNewColumns") 
        .option("header", "true")
        .option("badRecordsPath", BAD_RECORDS_PATH) # Production Error Handling
        .option("maxFilesPerTrigger", 1000) # Throttle processing
        .load(SOURCE_PATH))

    # --- 2. Extract Metadata ---
    bronze_with_meta = bronze_df.select("*", col("_metadata.file_path").alias("file_path")) \
        .withColumn("file_path", upper(col("file_path"))) \
        .withColumn("extraction_year", regexp_extract(col("file_path"), r"(\d{4})", 1).cast("int"))

    # --- 3. Compute Extraction Month ---
    month_expr = lit(None)
    for m_name, m_val in month_map.items():
        month_expr = when(col("file_path").contains(m_name), m_val).otherwise(month_expr)

    bronze_final = bronze_with_meta.withColumn("extraction_month", month_expr) \
        .withColumn("ingested_at", current_timestamp())

    # --- 4. Write to Bronze S3 Path ---
    query = (bronze_final.writeStream
        .format("delta")
        .option("checkpointLocation", CHECKPOINT_LOCATION) 
        .trigger(availableNow=True) # Production: Change to processingTime="1 hour" for continuous streams
        .start(BRONZE_S3_PATH))
    
    # Wait for the batch to finish
    query.awaitTermination()
    
    logger.info(f"Bronze Ingestion completed successfully to {BRONZE_S3_PATH}")

except AnalysisException as ae:
    logger.error(f"Spark Analysis schema error during ingestion: {ae}")
    # In production: Trigger an alert (Slack/Email/PagerDuty) here
    raise ae
except Exception as e:
    logger.error(f"FATAL ERROR in Bronze Layer: {e}")
    logger.error(traceback.format_exc())
    # In production: Trigger an alert
    raise e

